In [30]:
### section here is to download yfinance

import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"


####Note that the stock_config.json must be in the same directory is your py file

# Load stock config
with open("stock_config.json") as f:
    stock_config = json.load(f)

# Load BigQuery config
with open("bq_config.json") as f:
    bq_config = json.load(f)


##extract ticker values
ticker_symbols = stock_config["ticker"] 
period = stock_config["period"]



# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id"]


########## Peihan code section################################33
df = yf.download(ticker_symbols, period=period)
df.columns = ['{}_{}'.format(col[0], col[1]) for col in df.columns]  # flatten columns
df = df.reset_index()

df = (
    pd.melt(df, id_vars='Date', var_name='Price_Ticker', value_name='Value')
      .assign(Price_Type=lambda x: x.Price_Ticker.str.split('_').str[0],
              Ticker=lambda x: x.Price_Ticker.str.split('_').str[1])
      .drop(columns='Price_Ticker')
      .pivot_table(index=['Date', 'Ticker'], columns='Price_Type', values='Value')
      .reset_index()
)


##### Perhan Code Section ########################################


# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    df, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


/tmp/ipykernel_15369/493597677.py:40: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker_symbols, period=period)
[*********************100%***********************]  3 of 3 completed


LoadJob<project=meta-sanctum-461903-p3, location=US, id=f97fdf49-de6d-4bb6-b66b-ee81f1affa0a>

In [31]:
### Section here is to download fred data. Use the fred_config json to configure what you wish to pull out. 

import pandas_datareader.data as web
import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os


### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"

# Function to compute start_date based on period no longer need to specify end and 
def get_date_range_from_period(period_str):
    today = datetime.date.today()
    end_date = today

    if period_str.endswith("d"):
        delta = datetime.timedelta(days=int(period_str[:-1]))
    elif period_str.endswith("mo"):
        delta = relativedelta(months=int(period_str[:-2]))
    elif period_str.endswith("y"):
        delta = relativedelta(years=int(period_str[:-1]))
    else:
        raise ValueError(f"Invalid period: {period_str}. Use formats like '1mo', '3y', '7d'.")

    start_date = end_date - delta
    return datetime.datetime.combine(start_date, datetime.time.min), datetime.datetime.combine(end_date, datetime.time.min)

# Load configuration
with open("fred_config.json") as f:
    config = json.load(f)

with open("bq_config.json") as f:
    bq_config = json.load(f)




# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_fred"]

# Load series info
series_config = config["series"]



# Compute date range from period
period = config.get("period", "1y")  # default to 1 year if not specified
start_date, end_date = get_date_range_from_period(period)



# Download and combine data
econ_data = pd.DataFrame()

for series_code, series_name in series_config.items():
    print(f"Downloading {series_name} ({series_code})...")
    data = web.DataReader(series_code, 'fred', start_date, end_date)
    data.columns = [series_name]
    econ_data = data if econ_data.empty else econ_data.join(data, how='outer')











# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    econ_data, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


LoadJob<project=meta-sanctum-461903-p3, location=US, id=06cc4c9e-fe54-4c5f-80a6-5e8008cd6f63>

In [29]:
#### this code pulls all stock info which includes dividends and stuff sector

import pandas_datareader.data as web
import datetime
from dateutil.relativedelta import relativedelta
import yfinance as yf
import pandas as pd
from google.cloud import bigquery
import json
import os



### Importanting Service account json 
### note that your service account Json needs to be in the same folder as the py file you are using or you need to specify the file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "meta-sanctum-461903-p3-e443652134f2.json"



# Load configuration
with open("stock_config.json") as f:
    config = json.load(f)

with open("bq_config.json") as f:
    bq_config = json.load(f)


tickers = config.get("ticker", [])


# Extract BigQuery values
project_id = bq_config["project_id"]
dataset_id = bq_config["dataset_id"]
table_id = bq_config["table_id_stock_info"]



# List to collect info dicts
data = []

# Loop through each ticker
for symbol in tickers:
    try:
        ticker = yf.Ticker(symbol)
        info = ticker.get_info()
        if info:
            info["symbol"] = symbol  # Add symbol explicitly in case it's missing
            data.append(info)
        else:
            print(f"No data found for {symbol}")
    except Exception as e:
        print(f"Error retrieving {symbol}: {e}")

# Combine into one DataFrame
df_stock_info = pd.DataFrame(data)

# 3. Initialize BigQuery client
client = bigquery.Client(project=project_id)

# 4. Load data into BigQuery
job_config = bigquery.LoadJobConfig(
    ## section here is set to over ride the data. Using append complicates the code
    write_disposition="WRITE_TRUNCATE",  # or WRITE_TRUNCATE or # "Write_Append"
    autodetect=True,
    source_format=bigquery.SourceFormat.PARQUET
)

# Load DataFrame to BigQuery
job = client.load_table_from_dataframe(
    df_stock_info, f"{project_id}.{dataset_id}.{table_id}", job_config=job_config
)
job.result()  # Wait for completion  # or use .history(start=..., end=...


LoadJob<project=meta-sanctum-461903-p3, location=US, id=e03e58e1-a064-4189-aea9-1088f4b2750f>

In [25]:
df_stock_info

,address1,city,state,zip,country,phone,website,industry,industryKey,industryDisp,...,market,esgPopulated,postMarketChangePercent,postMarketPrice,postMarketChange,regularMarketChange,regularMarketDayRange,displayName,trailingPegRatio,address2
0,One Apple Park Way,Cupertino,CA,95014,United States,(408) 996-1010,https://www.apple.com,Consumer Electronics,consumer-electronics,Consumer Electronics,...,us_market,False,-0.316260,198.57,-0.629990,0.419998,197.3601 - 199.6799,Apple,1.8048,NaN
1,5205 North O'Connor Boulevard,Irving,TX,75039,United States,972 891 7700,https://www.caterpillar.com,Farm & Heavy Construction Machinery,farm-heavy-construction-machinery,Farm & Heavy Construction Machinery,...,us_market,False,-0.265955,360.00,-0.959991,-2.180020,357.87 - 362.0,Caterpillar,1.8763,Suite 100
2,929 Long Bridge Drive,Arlington,VA,22202-4208,United States,703-465-3500,https://www.boeing.com,Aerospace & Defense,aerospace-defense,Aerospace & Defense,...,us_market,False,0.098158,203.95,0.199997,-10.250000,201.28 - 206.32,NaN,NaN,NaN


# All code below this line is for exploration only

In [21]:
##df_stock_info.head()
df_subset = df_stock_info[['companyOfficers', 'sectorKey']]
df_subset

,companyOfficers,sectorKey
0,"[{'maxAge': 1, 'name': 'Mr. Timothy D. Cook', ...",technology
1,"[{'maxAge': 1, 'name': 'Mr. D. James Umpleby I...",industrials
2,"[{'maxAge': 1, 'name': 'Mr. Robert K. Ortberg'...",industrials


In [22]:
officer_cols = [col for col in df_stock_info.columns if 'company' in col]
print(officer_cols)


['companyOfficers']
